# 第 25 章 モデルの診断

Book 第 25 章に対応します。新しい方法を加えて，これまで見分けられなかった粒を見分けられるか調べます。最初に分類のブースティングで一段ずつ確率が動く仕組みを読みます。次に，同じ粒でモデルを替えたときの成功・新しい誤り・共通の誤りを確認し，次にどの試料や記録を調べるかを考えます。

上から順に実行してください。前半は講義の実演です。後半の実習（約 20 分）では，長軸の帯を一つ選び，同じ帯の未表示の二粒を形状から予想して，三つの方法の結果を読みます。

## 同じ粒と同じ規則

撮影後の七つの形状値で，Cammeo と Osmancik を識別します。Cammeo を 1，Osmancik を 0 とします。元行番号は照合に使い，入力は形状だけにそろえます。学習 2,286 粒，検証 762 粒は前章までと同じです。各品種 50 g の試料に由来する粒なので，別のロットでの性能は新しい試料で調べる必要があります。このノートは保留行を予測しません。

閾値は，方法の違いを比べるために 0.5 にそろえます。表示のために丸める前の Cammeo 確率で 0.5 以上なら候補に入れます。

In [ ]:
from pathlib import Path

from IPython.display import display
import numpy as np
import pandas as pd

local_candidates = [
    Path("data/rice-grains.csv"),
    Path("../data/rice-grains.csv"),
    Path("rice-grains.csv"),
]
local_csv = next((path for path in local_candidates if path.is_file()), None)
public_csv = (
    "https://raw.githubusercontent.com/sohei-takagi/"
    "python-ds-course-materials/main/data/rice-grains.csv"
)
rice = pd.read_csv(local_csv if local_csv is not None else public_csv)
features = [
    "area", "perimeter", "major_axis", "minor_axis", "eccentricity",
    "convex_area", "extent",
]
train = rice.loc[rice["split"] == "train"]
valid = rice.loc[rice["split"] == "validation"]
X_train, X_valid = train[features], valid[features]
y_train = train["variety"].eq("Cammeo").astype(int)
y_valid = valid["variety"].eq("Cammeo").astype(int)
threshold = 0.5

## 一段でスコアがどう変わるか

六行の架空の形状値で試します。正解は 0，0，1，0，1，1 なので，初期確率は 0.5，初期スコアは 0 です。最初の木は，学習行の `y - p` を手掛かりに形状値 2.5 で分かれ，対数損失に合わせた木の出力を左で −2，右で +1 とします。ここでは教材用に学習率 0.5 を使います。

`first_output` に 0.5 を掛けて初期スコアへ足し，更新した `score` を S 字に通して `probability` を得ます。形状値 4 は正解が 0 です。右葉のほかの行と同じ補正を受けるので，どちらに動きますか。

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

def sigmoid(score):
    return 1 / (1 + np.exp(-score))

toy_X = pd.DataFrame({"shape": [1, 2, 3, 4, 5, 6]})
toy_y = np.array([0, 0, 1, 0, 1, 1])
toy_model = GradientBoostingClassifier(
    n_estimators=2, learning_rate=0.5, max_depth=1, random_state=2026,
)
toy_model.fit(toy_X, toy_y)

toy_output = toy_model.estimators_[0, 0].predict(toy_X.to_numpy())
toy_score = 0 + toy_model.learning_rate * toy_output
toy_probability = sigmoid(toy_score)
toy_X.assign(y=toy_y, first_output=toy_output, score=toy_score,
             probability=toy_probability).round(3)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.5), layout="constrained")
for ax, values, boundary, ylabel in zip(
    axes, [toy_score, toy_probability], [0, 0.5], ["Score", "Cammeo probability"]
):
    ax.axhline(boundary, color="#737b83", linestyle="--", linewidth=1)
    ax.axvline(2.5, color="#737b83", linestyle=":", linewidth=1)
    for label, marker, color, name in [
        (0, "s", "#a84e23", "True class 0"), (1, "o", "#166179", "True class 1")
    ]:
        mask = toy_y == label
        ax.scatter(toy_X.loc[mask, "shape"], values[mask],
                   label=name, marker=marker, color=color, s=54, zorder=3)
    ax.set(xlabel="Shape value", ylabel=ylabel, xticks=range(1, 7), xlim=(0.6, 6.4))
axes[0].set_ylim(-1.32, 0.83)
axes[1].set_ylim(0, 1)
axes[0].legend(loc="upper left", fontsize=9)
plt.show()

## 実際の二粒を追う

前章と同じ七列ロジスティック回帰（LR）とランダムフォレスト（森）に，分類の勾配ブースティング（GB）を加えます。GB は深さ 2 以下の木を 200 段，学習率 0.05 で加えます。この設定で学習する行は `X_train` と `y_train` です。

次に，検証の元行 1086 と 3799 に第一段の木を当てます。学習で Cammeo は 978 粒なので，初期確率は `978 / 2286` です。`First tree output` に学習率を掛け，初期スコアへ足してから S 字に通します。最初の木が返す出力は，浅い分類木の葉に入った Cammeo の割合とは異なる数です。

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

logistic = Pipeline([
    ("scale", StandardScaler()),
    ("logistic", LogisticRegression(C=1.0, max_iter=2000)),
])
forest = RandomForestClassifier(
    n_estimators=200, min_samples_leaf=5, max_features="sqrt",
    random_state=2026, n_jobs=2,
)
boosting = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=2,
    random_state=2026, loss="log_loss",
)
models = {"LR": logistic, "Forest": forest, "GB": boosting}
for model in models.values():
    model.fit(X_train, y_train)

In [ ]:
example_ids = [1086, 3799]
examples = valid.set_index("record_id").loc[example_ids]
X_examples = examples[features]

initial_probability = y_train.mean()
initial_score = np.log(initial_probability / (1 - initial_probability))
first_output = boosting.estimators_[0, 0].predict(X_examples.to_numpy())
first_score = initial_score + boosting.learning_rate * first_output
pd.DataFrame({
    "Initial probability": initial_probability,
    "Initial score": initial_score,
    "First tree output": first_output,
    "Updated score": first_score,
    "Updated probability": sigmoid(first_score),
}, index=examples.index).round(4)

`staged_predict_proba()` で，学習を済ませた GB を一段ずつ使った確率を読みます。行 0 は木を使う前の初期確率で，1 以降は使う木の本数です。実際は 1086 が Cammeo，3799 が Osmancik です。二粒について 0.5 を超えるかを確認し，最後にどう判定するかを比べます。

In [ ]:
assert boosting.classes_.tolist() == [0, 1]

probability_steps = np.array([
    probabilities[:, 1] for probabilities in boosting.staged_predict_proba(X_examples)
])
paths = pd.DataFrame(
    probability_steps, index=range(1, boosting.n_estimators + 1), columns=example_ids,
)
paths.loc[0] = initial_probability
paths = paths.sort_index().rename_axis("Trees")
paths.loc[[0, 1, 10, 50, 200]].round(5)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.8, 3.65), layout="constrained")
ax.plot(paths.index, paths[1086], color="#166179", linewidth=2, label="1086 · Cammeo")
ax.plot(paths.index, paths[3799], color="#a84e23", linewidth=2, linestyle="--",
        label="3799 · Osmancik")
ax.axhline(0.5, color="#737b83", linestyle=":", linewidth=1.4, label="Threshold")
ax.set(xlabel="Trees used", ylabel="Cammeo probability", xlim=(0, 200), ylim=(0, 1))
ax.legend(loc="lower right", fontsize=9)
plt.show()

## 全粒で正誤を合わせる

`predictions` の一行は検証の一粒です。三つの方法の Cammeo 確率を `p_` で始まる列に，0.5 での判定を `predict_` で始まる列に入れます。`correct_` は判定が正解と同じなら `True` です。

一つ目の表は前章と同じく，品種の四件数を数えます。二つ目の対応表は，LR と GB の正誤を同じ粒で照合します。左下の件数は GB が直した粒，右上は GB が新しく外した粒です。両方を使って全体の正解数の差を説明できますか。

In [ ]:
predictions = valid[["record_id", "variety", "major_axis"]].copy()
predictions["y"] = y_valid

for name, model in models.items():
    assert model.classes_.tolist() == [0, 1]
    predictions[f"p_{name}"] = model.predict_proba(X_valid)[:, 1]
    predictions[f"predict_{name}"] = (predictions[f"p_{name}"] >= threshold).astype(int)
    predictions[f"correct_{name}"] = predictions[f"predict_{name}"].eq(predictions["y"])

In [ ]:
from sklearn.metrics import confusion_matrix

def count_decisions(rows):
    counts = {}
    for name in models:
        tn, fp, fn, tp = confusion_matrix(
            rows["y"], rows[f"predict_{name}"], labels=[0, 1],
        ).ravel()
        counts[name] = {"Correct": tn + tp, "TN": tn, "FP": fp, "FN": fn, "TP": tp}
    return pd.DataFrame.from_dict(counts, orient="index")

count_decisions(predictions)

In [ ]:
def compare_correctness(rows):
    paired = pd.crosstab(rows["correct_LR"], rows["correct_GB"])
    paired = paired.reindex(index=[True, False], columns=[True, False], fill_value=0)
    paired.index = ["LR correct", "LR wrong"]
    paired.columns = ["GB correct", "GB wrong"]
    return paired

compare_correctness(predictions)

## 長軸の帯で読む

`band` は 180 未満（`under180`），180 以上 200 未満（`180to200`），200 以上（`200plus`）です。`right=False` で右端を次の帯に入れます。`Always Osmancik correct` は，第 23 章で使った，全粒を Osmancik と判定する基準の正解数です。選んだ帯の品種の内訳と比べ，形状を使う三つの方法が何粒を正しく識別したか読みます。

図には各帯の粒数を載せました。中央帯の GB の誤り 51 粒は，この帯での誤りの割合を求めるとき，何で割りますか。

In [ ]:
band_names = ["under180", "180to200", "200plus"]
predictions["band"] = pd.cut(
    predictions["major_axis"], bins=[-np.inf, 180, 200, np.inf],
    labels=band_names, right=False,
)

by_band = predictions.groupby("band", observed=True)
band_summary = by_band.agg(Grains=("y", "size"), Cammeo=("y", "sum"))
band_summary["Always Osmancik correct"] = band_summary["Grains"] - band_summary["Cammeo"]
for name in models:
    band_summary[f"{name} correct"] = by_band[f"correct_{name}"].sum()
band_summary

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8.3, 4.1), layout="constrained")
x = np.arange(len(band_summary))
grains = band_summary["Grains"]
for index, (name, color, hatch) in enumerate([
    ("LR", "#1a637a", ""), ("Forest", "#9e522e", "/"), ("GB", "#497556", "."),
]):
    error_rate = 100 * (grains - band_summary[f"{name} correct"]) / grains
    ax.bar(x + (index - 1) * 0.23, error_rate, width=0.22,
           label=name, color=color, edgecolor="#363d42", hatch=hatch)
ax.set(xticks=x, xticklabels=[
    "< 180\n(n = 305)", "180 to < 200\n(n = 221)", "≥ 200\n(n = 236)",
], xlabel="Major axis length", ylabel="Errors within band (%)", ylim=(0, 27))
ax.set_axisbelow(True)
ax.grid(axis="y", color="#d9dddf", linewidth=0.7)
ax.legend(loc="upper left", ncol=3, fontsize=9)
plt.show()

## 四粒を読む

予測表から元行番号で四粒を取り出します。1086 は両方正しく，3799 は両方が誤って候補に入れました。3385 と 1584 は，LR から GB に替えるとどちらも 0.5 より下に動きます。どちらで改善し，どちらで新しい誤りが生じましたか。図では二つの確率を座標にし，同じ粒が閾値のどちら側にあるかを確かめます。

In [ ]:
inspect_ids = [1086, 3799, 3385, 1584]
predictions.set_index("record_id").loc[inspect_ids, [
    "variety", "major_axis", "p_LR", "p_GB", "predict_LR", "predict_GB",
]].round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.7, 5.25), layout="constrained")
for variety, marker, color in [
    ("Osmancik", "s", "#a84e23"), ("Cammeo", "o", "#166179"),
]:
    group = predictions.loc[predictions["variety"] == variety]
    ax.scatter(group["p_LR"], group["p_GB"], label=variety,
               marker=marker, color=color, s=17, alpha=0.50, linewidths=0)
ax.axvline(threshold, color="#737b83", linestyle="--", linewidth=1)
ax.axhline(threshold, color="#737b83", linestyle="--", linewidth=1)
for record_id, offset in [(3385, (1, -30)), (1584, (32, -16))]:
    grain = predictions.set_index("record_id").loc[record_id]
    point = (grain["p_LR"], grain["p_GB"])
    ax.scatter(*point, color="none", edgecolor="#1f262b", s=76, linewidths=1.1, zorder=3)
    ax.annotate(str(record_id), point, xytext=offset, textcoords="offset points",
                fontsize=9, arrowprops={"arrowstyle": "-", "color": "#1f262b"})
ax.set(xlabel="LR · Cammeo probability", ylabel="GB · Cammeo probability",
       xlim=(-0.03, 1.03), ylim=(-0.03, 1.03))
ax.legend(loc="upper left", fontsize=9)
plt.show()

## 次に確かめること

同じ帯の誤りが多くても，原因はまだ分かりません。形状値が写真によって変わるなら，同じ粒を再撮影して照合します。二品種の形状が安定して重なるという説明と区別するために，その帯の成功例・誤り例の両方を含めます。別のロットでの性能を調べるなら，両品種の別ロットを集め，ロットや装置を記録して，学習側にないロットで比べます。本章の図と表は，次の調査を考える材料です。

## 実習（20 分）

### 1．帯を選んで絞る（約 8 分）

| `selected_band` に入れる文字列 | 長軸の条件 | グループ A の候補 | グループ B の候補 |
|---|---|---|---|
| `"under180"` | 180 未満 | `"A1"`，`"A2"` | `"B1"`，`"B2"` |
| `"180to200"` | 180 以上 200 未満 | `"A1"`，`"A2"` | `"B1"`，`"B2"`，`"B3"` |
| `"200plus"` | 200 以上 | `"A1"`，`"A2"` | `"B1"`，`"B2"` |

どの帯のどんな違いを調べたいか，講義で見た形状の分布や正解数を手掛かりに一文で書きます。

**選んだ帯と理由**　（ここに記入）

次のセルで `selected_band` を選びます。`band_rows = None` の右辺を書き換え，`predictions` から `"band"` 列が `selected_band` に等しい行を `.loc[]` で取り出してください。まず境の値ちょうどがどの帯に入るかを確かめ，セルを実行して粒数を確認します。

### 2．二粒を予想して照合する（約 6 分）

同じセルで `selected_a` と `selected_b` を選びます。`show_results = False` のまま実行すると，選んだ二粒の形状だけが出ます。それぞれについて，講義で見た形状の分布を使って，どちらの品種らしいか，または自信を持って見分けにくいかを考えます。予想と根拠をセルの下へ書いてから，`show_results = True` に変えて同じセルを実行します。選んだ二粒とその帯の結果が表示されます。

In [ ]:
selected_band = None  # "under180", "180to200", "200plus"
selected_a = None     # "A1" or "A2"
selected_b = None     # "B1", "B2", or "B3" where available
show_results = False

practice_candidates = {
    "under180": ({"A1": 1649, "A2": 1665}, {"B1": 1019, "B2": 3248}),
    "180to200": ({"A1": 10, "A2": 34}, {"B1": 1510, "B2": 3644, "B3": 136}),
    "200plus": ({"A1": 8, "A2": 12}, {"B1": 1646, "B2": 1670}),
}

if selected_band not in practice_candidates:
    print("表から長軸の帯を一つ選んでください。")
else:
    band_rows = None  # predictions から選んだ帯だけを取り出す
    expected_band = predictions["band"].eq(selected_band)
    if band_rows is None:
        print("band_rows の右辺を，選んだ帯を抽出する式に書き換えてください。")
    elif not band_rows.index.equals(predictions.index[expected_band]):
        print("長軸の帯で絞り，選んだ帯の全行が元の順序で残るかを確認してください。")
    else:
        print(f"帯: {selected_band} · {len(band_rows)} 粒")
        candidates_a, candidates_b = practice_candidates[selected_band]
        if selected_a not in candidates_a or selected_b not in candidates_b:
            print(f"候補を一つずつ選んでください。A: {list(candidates_a)} / B: {list(candidates_b)}")
        else:
            selected_ids = [candidates_a[selected_a], candidates_b[selected_b]]
            chosen = band_rows.set_index("record_id").loc[selected_ids]
            if not show_results:
                display(valid.set_index("record_id").loc[chosen.index, features]
                        .set_axis([selected_a, selected_b]).round(4))
            else:
                n_cammeo = band_rows["y"].sum()
                print(f"閾値: {threshold} · Cammeo: {n_cammeo} 粒 · Osmancik: {len(band_rows) - n_cammeo} 粒")
                print("帯内の四件数")
                display(count_decisions(band_rows))
                print("帯内の正誤対応")
                display(compare_correctness(band_rows))
                print("全検証行の四件数")
                display(count_decisions(predictions))
                print(f"二粒の予測と正解: {selected_a}, {selected_b} の順")
                display(chosen[[
                    "variety", "major_axis", "p_LR", "p_Forest", "p_GB",
                    "predict_LR", "predict_Forest", "predict_GB",
                ]].round(4))

**二粒についての予想と根拠**　（ここに記入。結果を表示した後も残す）

### 3．判断と次の比較を書く（約 6 分）

1. 成功した一粒と，方法によって判定が分かれたか共通して外れた一粒を述べます。元行番号，実際の品種，表示された確率と判定を使います。帯内に LR と GB の不一致がない場合は，共通して外れた粒を読みます。
2. 選んだ帯の正誤対応表と全 762 粒の四件数を根拠に，今回の規則でどの方法を保つか，追加するかを 3〜4 文で述べます。
3. その判断を見直すには，次に何をどんな条件で比べるかを一つ書きます。

**報告**　（ここに記入）

帯や二粒を変える場合は，`show_results = False` に戻して同じセルを実行し，改めて形状から理由を考えます。結果を出したら，出力の帯と元行番号を確かめます。選択理由と予想，選んだ帯の件数と正誤の対応表，二粒の結果，報告が残るように，カーネルを再起動し，上から全セルを実行して提出してください。

### データの出典

Cinar, I., & Koklu, M. (2019). *Rice (Cammeo and Osmancik)*. UCI Machine Learning Repository. 教材用 CSV は列名を短縮し，元の行番号と授業用の分割を追加しています。 [データ配布元](https://doi.org/10.24432/C5MW4Z) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) ・ [原著論文](https://doi.org/10.18201/ijisae.2019355381)